In [1]:
# from IPython.core.interactiveshell import InteractiveShell
# InteractiveShell.ast_node_interactivity = "all"
import pandas as pd
import seaborn as sns
sns.set()
import gc
gc.collect()
%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import set_matplotlib_formats
from IPython.display import clear_output
set_matplotlib_formats('retina')
#from tqdm import tqdm
#tqdm.pandas()
import numpy as np
from pyhive import presto
from datetime import datetime, timedelta
# from bson import ObjectId
from functools import reduce
from sklearn.cluster import KMeans
# from pymongo import MongoClient
import glob
import warnings
warnings.filterwarnings('ignore')
from datetime import date
import json
import re
#import dtale
#from h3 import h3
# import pandasql as ps
import datetime
import statsmodels.api as sm
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.seasonal import seasonal_decompose

/var/folders/fb/2nwyf_mx609gfjxv_jjnb76h0000gn/T/ipykernel_12925/1857937564.py:12: DeprecationWarning: `set_matplotlib_formats` is deprecated since IPython 7.23, directly use `matplotlib_inline.backend_inline.set_matplotlib_formats()`
  set_matplotlib_formats('retina')


In [5]:
metabase_connection = presto.connect(
        # host='bi-trino-2.serving.data.production.internal',
#         host='bi-trino.serving.data.production.internal',
        #host='bi-presto.serving.data.production.internal',
        #host='prime-trino.serving.data.production.internal',
         host="processing.processing.data.production.internal",
         
#         host="presto.processing.yoda.run",
        # host="processing.processing.data.production.internal",


        port=80,
        protocol='http',
        catalog ='hive',
        username='sameer.kumar@rapido.bike')
    

In [14]:
q="""
with captain_single_view as ( 
select distinct captainid captain_id, mobilenumber
from datasets.captain_single_view
where lower(lastridecity)='chennai'
),seg as (
    select 
        distinct captain_id, performance_segment, consistency_segment
    from datasets_internal.captain_subservice_segments_v1
    where lower(service_mode)= 'all'
    and lower(service) = 'link'
    and lower(city) in ('chennai')
    and run_date = '20250223'
),
gigStatus as(
select captain_id,gig_segment
from ( 
        select *,row_number() over(partition by captain_id order by run_date desc) as r
        from reports_internal.captain_gig_appography_v4
        where run_date = '20250107'
        and lower(city) in ('chennai')
    )

where r=1
), subs_caps as (
    select userid
    from canonical.user_selector 
    where yyyymmdd='20250222'
    and usertype = 'rider'
    and (
        name like '%BLR_Sub_not_offered_EBNP_29_%'
        or name like '%BLR_subs_purachased_captain_%'
        or name like '%BLR_earning_base_purcahsed_%'
        or name like '%BLR_subs not offered_ride_29_%'
        or name like '%BLR_inactive_subs_11Rs_Fri_1_%'
        or name like '%BLR_inactive_subs_11Rs_Fri_%'
        or name like '%BLR_inactive_subs_11Rs_Fri_2_%'
    )

)
select a.*, c.mobilenumber,d.performance_segment, d.consistency_segment,b.gig_segment
from (
   select captain_id,sum(net_rides_taxi) as net_rides, sum(total_lh) as total_lh, sum(final_captain_earnings) as total_earnings, sum(total_incentives) as total_incentives,count(case when net_rides_taxi>0 then yyyymmdd end) as net_days
   from reports_internal.consideration_temp_captain_logs_base
   where yyyymmdd between '20250204' and '20250218'
   and lower(city) in ('bangalore')
   group by 1
) a
left join gigStatus b
on a.captain_id=b.captain_id
left join captain_single_view c
on c.captain_id = c.captain_id
left join seg d
on d.captain_id=a.captain_id
where a.net_rides>0
and a.captain_id in(select userid from subs_caps)
"""
bangloreBase=pd.read_sql(q,metabase_connection)

ConnectTimeout: HTTPConnectionPool(host='processing.processing.data.production.internal', port=80): Max retries exceeded with url: /v1/statement/executing/20250224_192710_00273_heb37/y1a5e207229586fbc10cae5ae1707ff3b9e4b9776/10884 (Caused by ConnectTimeoutError(<urllib3.connection.HTTPConnection object at 0x153bcc440>, 'Connection to processing.processing.data.production.internal timed out. (connect timeout=None)'))

In [34]:
q="""
    select userid
    from canonical.user_selector 
    where yyyymmdd='20250222'
    and usertype = 'rider'
    and (
        name like '%BLR_Sub_not_offered_EBNP_29_%'
        or name like '%BLR_subs_purachased_captain_%'
        or name like '%BLR_earning_base_purcahsed_%'
        or name like '%BLR_subs not offered_ride_29_%'
        or name like '%BLR_inactive_subs_11Rs_Fri_1_%'
        or name like '%BLR_inactive_subs_11Rs_Fri_%'
        or name like '%BLR_inactive_subs_11Rs_Fri_2_%'
    )
"""
bangloreBase=pd.read_sql(q,metabase_connection)

In [ ]:
q="""
    select captain_id,sum(net_rides_taxi) as net_rides, sum(total_lh) as total_lh, sum(final_captain_earnings) as total_earnings, sum(total_incentives) as total_incentives,count(case when net_rides_taxi>0 then yyyymmdd end) as net_days
   from reports_internal.consideration_temp_captain_logs_base
   where yyyymmdd between '20250208' and '20250222'
   and lower(city) in ('bangalore')
   and net_rides_taxi>0
   group by 1
"""
bangaloreData=pd.read_sql(q,metabase_connection)

In [36]:
q="""
    select captainid captain_id, mobilenumber
    from datasets.captain_single_view
    where lower(lastridecity)='bangalore'
"""
captainphone=pd.read_sql(q,metabase_connection)

In [37]:
q="""
  select 
            distinct captain_id, performance_segment, consistency_segment
        from datasets_internal.captain_subservice_segments_v1
        where lower(service_mode)= 'all'
        and lower(service) = 'link'
        and lower(city) in ('bangalore')
        and run_date = '20250223'
"""
segment=pd.read_sql(q,metabase_connection)

In [38]:
q="""
 select distinct captain_id,gig_segment
      from ( 
            select *
            from reports_internal.captain_gig_appography_v4
            where run_date = '20250107'
            and lower(city) in ('bangalore')
          )
"""
gigStatus=pd.read_sql(q,metabase_connection)

In [39]:
bangloreBase.columns=['captain_id']

In [40]:
bangaloreData=pd.merge(bangaloreData,gigStatus,how='left',on='captain_id')
bangaloreData=pd.merge(bangaloreData,segment,how='left',on='captain_id')
bangaloreData=pd.merge(bangaloreData,captainphone,how='left',on='captain_id')
bangaloreData=pd.merge(bangaloreData,bangloreBase,how='inner',on='captain_id')

In [44]:
len(bangloreBase)

34832

In [42]:
bangaloreData['net_rides_cat']=pd.qcut(bangaloreData['net_rides'],3,['A','B','C'])
bangaloreData['total_lh_cat']=pd.qcut(bangaloreData['total_lh'],3,['D','E','F'])
bangaloreData['total_earnings_cat']=pd.qcut(bangaloreData['total_earnings'],3,['H','I','J'])
bangaloreData['net_days_cat']=pd.qcut(bangaloreData['net_days'],3,['K','L','M'])

In [45]:
bangaloreData

,captain_id,net_rides,total_lh,total_earnings,total_incentives,net_days,gig_segment,performance_segment,consistency_segment,mobilenumber,net_rides_cat,total_lh_cat,total_earnings_cat,net_days_cat
0,65b48ffe1ca8733054dee67f,20.0,30.0318,3482.194998,0.0,6,10_RHA_gig,LP,Intra,7073519158,C,F,J,M
1,67756bc74d2d5dcd3049ab67,53.0,21.6461,3269.671200,40.0,7,11_Full_Gig,MP,Inter,9110852217,C,F,J,M
2,677f191e5f97a53cac72136a,48.0,50.3872,5903.026958,0.0,11,NaN,LP,Intra,7996859949,C,F,J,M
3,67977f56a1e5511d579ac5fe,34.0,19.7530,1697.138693,0.0,6,NaN,LP,Intra,9566964985,C,F,J,M
4,5d2840f8668011467e2d45da,32.0,12.6651,1398.499425,200.0,9,00_non_Gig,LP,Inter,9739849965,C,F,J,M
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25269,62ea0257d5c1cb2c8995f02f,2.0,3.7558,166.638095,0.0,1,10_RHA_gig,LP,Sporadic,9439114347,A,E,H,K
25270,663f419354d49f7ff37a1626,1.0,4.4574,144.094389,0.0,1,00_non_Gig,LP,Sporadic,9395373042,A,E,H,K
25271,663f419354d49f7ff37a1626,1.0,4.4574,144.094389,0.0,1,00_non_Gig,LP,Sporadic,9395373042,A,E,H,K
25272,647ac6523294844de4df4858,4.0,4.2383,376.837528,0.0,1,11_Full_Gig,LP,Sporadic,8310741039,A,E,I,K


In [32]:
bangaloreData['total_incentives_cat']=pd.qcut(bangaloreData['total_incentives'],3,['N'],duplicates='drop')

In [48]:
bangaloreData.gig_segment.fillna('NA',inplace=True)

In [49]:
df=bangaloreData

In [50]:
strata_groups = ['gig_segment', 'performance_segment', 'consistency_segment', 'net_rides_cat', 'total_lh_cat','total_earnings_cat', 'net_days_cat']
count_group = "captain_id"
split_ratio = 0.5

# age = np.random.randint(low=18, high=60, size=1000)
# income = np.random.randint(low=18000, high=60000, size=1000)

# age_bucket = pd.qcut(x=age, q=[0,0.25,0.50,0.75,1],labels=["P<0.25","P0.25_0.5","P0.5_0.75","P_0.75+"])
# income_bucket = pd.qcut(x=income, q=[0,0.25,0.50,0.75,1],labels=["P<0.25","P0.25_0.5","P0.5_0.75","P_0.75+"])

# df = pd.DataFrame({'age':age,"income":income, "age_group": age_bucket, 'income_group': income_bucket})
# df

whole_set = df.index.values

np.random.seed(100)
stratified_sample = df.groupby(strata_groups).apply(lambda x: x.sample(frac=split_ratio,random_state=100))
strata= stratified_sample.droplevel(list(range(len(strata_groups))))

test_group = np.sort(strata.index.values)
control_group = np.sort(np.delete(arr=whole_set, obj=test_group,axis=None))

test_df = df.iloc[test_group]
control_df = df.iloc[control_group]
test_df.shape, control_df.shape

def split_check_func(split_input_df, data_type):
    split_check = split_input_df.groupby(strata_groups).agg({count_group: ['count']})
    for i in range(len(strata_groups)):
        split_check.insert(loc=i,column= split_check.index.names[i], value=split_check.index.get_level_values(i))
    split_check = split_check.reset_index(drop=True)
    split_check.columns= ["_".join(split_check.columns[i]).rstrip("_") for i in range(len(split_check.columns))]
    column_name = [i for i in split_check.columns if "_count" in i]
    new_column = data_type+"_"+column_name[0]
    print(column_name, new_column)
    split_check = split_check.rename(columns={column_name[0]: new_column})
    print(data_type,split_check[new_column].sum())
    split_check[data_type+'_perc'] = split_check[new_column]/ np.sum(split_check[new_column])
    return(split_check)

test_split = split_check_func(test_df, "test")
control_split = split_check_func(control_df,"control")

test_control_df = pd.merge(left=test_split, right=control_split, on= strata_groups, how='inner')
# test_control_df.to_clipboard()

test_col = 'test_{}_count'.format(count_group)
control_col = 'control_{}_count'.format(count_group)

test_control_df["split_ratio_check"] = np.round(test_control_df[test_col]/(test_control_df[test_col]+ test_control_df[control_col]),2)
print(np.quantile(test_control_df['split_ratio_check'], q=[0.05,0.25,0.5,0.75,0.95]))
test_control_df


['captain_id_count'] test_captain_id_count
test 12558
['captain_id_count'] control_captain_id_count
control 12716
[nan nan nan nan nan]


,gig_segment,performance_segment,consistency_segment,net_rides_cat,total_lh_cat,total_earnings_cat,net_days_cat,test_captain_id_count,test_perc,control_captain_id_count,control_perc,split_ratio_check
0,00_non_Gig,HP,Daily,A,D,H,K,0,0.00000,0,0.000000,NaN
1,00_non_Gig,HP,Daily,A,D,H,L,0,0.00000,0,0.000000,NaN
2,00_non_Gig,HP,Daily,A,D,H,M,0,0.00000,0,0.000000,NaN
3,00_non_Gig,HP,Daily,A,D,I,K,0,0.00000,0,0.000000,NaN
4,00_non_Gig,HP,Daily,A,D,I,L,0,0.00000,0,0.000000,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
6475,NA,UHP,Sporadic,C,F,I,L,0,0.00000,0,0.000000,NaN
6476,NA,UHP,Sporadic,C,F,I,M,0,0.00000,0,0.000000,NaN
6477,NA,UHP,Sporadic,C,F,J,K,1,0.00008,1,0.000079,0.5
6478,NA,UHP,Sporadic,C,F,J,L,0,0.00000,0,0.000000,NaN


In [52]:
test_df.reset_index(drop='index',inplace=True)
control_df.reset_index(drop='index',inplace=True)

In [53]:
test_df.to_csv('bangalore_test_df.csv')
control_df.to_csv('bangalore_control_df.csv')